In [0]:
%sql
-- CREATE SCHEMA IF NOT EXISTS dbcertification.easewithdata
-- COMMENT 'Schema for ease with data'
-- LOCATION 'abfss://data@dbcertificationstg.dfs.core.windows.net/easewithdata';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbcertification.easewithdata.orders_raw_refactored DEEP CLONE samples.tpch.orders
LOCATION 'abfss://data@dbcertificationstg.dfs.core.windows.net/easewithdata/orders_raw_refactored';

CREATE TABLE IF NOT EXISTS dbcertification.easewithdata.customer_raw_refactored DEEP CLONE samples.tpch.customer
LOCATION 'abfss://data@dbcertificationstg.dfs.core.windows.net/easewithdata/customer_raw_refactored';

In [0]:
%sql
select * from dbcertification.easewithdata.orders_raw_refactored limit 5;

In [0]:
%sql
select * from dbcertification.easewithdata.customer_raw_refactored limit 5;

#### Comparison between og and refactored table

In [0]:
%sql
select * from dbcertification.easewithdata.orders_agg_gold;

In [0]:
%sql
select * from dbcertification.easewithdata.orders_agg_gold_refactored;

**Add some data incrementally in ORDERS**

In [0]:
%sql
INSERT INTO dbcertification.easewithdata.orders_raw_refactored
SELECT * FROM samples.tpch.orders
LIMIT 10000;

In [0]:
%sql
select * from dbcertification.easewithdata.joined_silver_refactored

In [0]:
%sql
select * from dbcertification.easewithdata.orders_agg_gold;

In [0]:
%sql
select * from dbcertification.easewithdata.orders_agg_gold_refactored;

In [0]:
%sql
select * from dbcertification.easewithdata.orders_agg_gold;

In [0]:
%sql
DESCRIBE HISTORY dbcertification.easewithdata.orders_bronze_refactored;

#### DLT with AutoLoader

In [0]:
dbutils.fs.mkdirs("/Volumes/dbcertification/easewithdata/volume_easewithdata/dlt_autoloader_landing_refactored/")

In [0]:
dbutils.fs.mkdirs("/Volumes/dbcertification/easewithdata/volume_easewithdata/dlt_autoloader_landing_refactored/files")
dbutils.fs.mkdirs("/Volumes/dbcertification/easewithdata/volume_easewithdata/dlt_autoloader_landing_refactored/autoloader/schemas")

#### SCD Type 1 and Type 2 in DLT - Changes to be done in cutomer_bronze table

In [0]:
%sql
ALTER TABLE dbcertification.easewithdata.customer_raw_refactored
ADD COLUMNS
(
    __src_action STRING, -- Possible values Insert, Delete, Truncate
    __src_insert_dt TIMESTAMP
);
-- Any changes in data at the source side will be a new record appended into the source table

In [0]:
%sql
-- update those columns added in table 
UPDATE dbcertification.easewithdata.customer_raw_refactored
SET __src_action = 'I', 
    __src_insert_dt = current_timestamp() - interval '3 days';

In [0]:
%sql
-- display the table
SELECT * FROM dbcertification.easewithdata.customer_raw_refactored;

In [0]:
%sql
select * from dbcertification.easewithdata.customer_scd1_bronze_refactored
where c_custkey = 6;

In [0]:
%sql
select * from dbcertification.easewithdata.customer_scd2_bronze_refactored
where c_custkey = 6;

In [0]:
%sql

INSERT INTO dbcertification.easewithdata.customer_raw_refactored
VALUES(
   6, 'Customer#000412450',	'fUD6IoGdtF',20,'30-293-696-4951',4406.28,'BUILDING','New Changed Record','I',current_timestamp()
);

In [0]:
%sql
select * from dbcertification.easewithdata.customer_scd1_bronze_refactored
where c_custkey = 6;

In [0]:
%sql
select * from dbcertification.easewithdata.customer_scd2_bronze_refactored
where c_custkey = 6;

#### BACK LOADING RECORD

In [0]:
%sql

INSERT INTO dbcertification.easewithdata.customer_raw_refactored
VALUES(
   6, 'Customer#000412450',	'fUD6IoGdtF',20,'30-293-696-4951',4406.28,'BUILDING','Old Record for BAckloading','I',current_timestamp()- interval '1 days'
);

- Backload doesnt happen for scd type 1 table

In [0]:
%sql
select * from dbcertification.easewithdata.customer_scd2_bronze_refactored
where c_custkey = 6
order by `__START_AT`;

- **DELETE** in SCD

In [0]:
%sql

INSERT INTO dbcertification.easewithdata.customer_raw_refactored
VALUES(
   6, 'Customer#000412450',	'fUD6IoGdtF',20,'30-293-696-4951',4406.28,'BUILDING','New Changed Record','D',current_timestamp()
);

In [0]:
%sql
select * from dbcertification.easewithdata.customer_scd1_bronze_refactored
where c_custkey = 6;

In [0]:
%sql
-- No impact as delete feature was not added
select * from dbcertification.easewithdata.customer_scd2_bronze_refactored
where c_custkey = 6;

**TRUNCATE** Table

In [0]:
%sql

INSERT INTO dbcertification.easewithdata.customer_raw_refactored
VALUES(
   null, 'Customer#000412450',	'fUD6IoGdtF',20,'30-293-696-4951',4406.28,'BUILDING','New Changed Record','T',current_timestamp()
);

In [0]:
%sql
select * from dbcertification.easewithdata.customer_scd1_bronze_refactored
-- whole table truncated